## Analytical Derivation of the Recombination

In [1]:
from sympy import *
import numpy as np
import matplotlib.pyplot as plt

In [2]:
𝕚 = I

In [3]:
Δϕ, Δp, Δθ, θ, p = symbols("Δϕ, Δp, ΔΘ, Theta, p", positive=True)
Ψ0 = symbols("Psi_0", real=True)
Ψa = symbols("Psi_+")
Ψb = symbols("Psi_-")

The initial wave function is

In [4]:
Ψinit = Matrix([
    [Ψ0],
    [0]
])

In [5]:
Ψinit

Matrix([
[Psi_0],
[    0]])

We apply a π/2 pulse:

In [6]:
U_split = Matrix(
    [[1, 𝕚], [𝕚, 1]]
) / sqrt(2)
U_split

Matrix([
[  sqrt(2)/2, sqrt(2)*I/2],
[sqrt(2)*I/2,   sqrt(2)/2]])

and introduce a relative phase

In [7]:
Φ = Matrix(
    [[exp(-𝕚*Δϕ/2), 0], [0, exp(+𝕚*Δϕ/2)]]
)
Φ

Matrix([
[exp(-I*Δϕ/2),           0],
[           0, exp(I*Δϕ/2)]])

to get the state before the recombination:

In [8]:
Ψ = Φ @ (U_split @ Ψinit)
Ψ

Matrix([
[ sqrt(2)*Psi_0*exp(-I*Δϕ/2)/2],
[sqrt(2)*I*Psi_0*exp(I*Δϕ/2)/2]])

Note that this is normalized:

In [9]:
def inner(a, b):
    return a.dot(b, hermitian=True, conjugate_convention="left")

In [10]:
def norm(Ψ):
    return sqrt(inner(Ψ, Ψ)).expand()

In [11]:
norm(Ψ)

Abs(Psi_0)

We also allow the wave packet $\Psi_0$ to evolve differently

In [12]:
Ψ = Matrix([
    [Ψ[0,0].subs({Ψ0: Ψa})],
    [Ψ[1,0].subs({Ψ0: Ψb})],
])
Ψ

Matrix([
[ sqrt(2)*Psi_+*exp(-I*Δϕ/2)/2],
[sqrt(2)*I*Psi_-*exp(I*Δϕ/2)/2]])

We then recombine by applying an inverse π/2 pulse:

In [13]:
U_recomb = U_split.conjugate().transpose()
U_recomb

Matrix([
[   sqrt(2)/2, -sqrt(2)*I/2],
[-sqrt(2)*I/2,    sqrt(2)/2]])

In [14]:
Ψ_recomb = U_recomb @ Ψ
Ψ_recomb

Matrix([
[     Psi_+*exp(-I*Δϕ/2)/2 + Psi_-*exp(I*Δϕ/2)/2],
[-I*Psi_+*exp(-I*Δϕ/2)/2 + I*Psi_-*exp(I*Δϕ/2)/2]])

In [15]:
norm(Ψ_recomb).expand()

sqrt(Psi_+*conjugate(Psi_+)/2 + Psi_-*conjugate(Psi_-)/2)

The complex amplitude on the "right"/"+" surface (index 1) is

## Amplitude and population for "+" surface

In [16]:
a = Ψ_recomb[0,0]

a

Psi_+*exp(-I*Δϕ/2)/2 + Psi_-*exp(I*Δϕ/2)/2

Which corresponds to the population

In [17]:
pop_a = (a * a.conjugate()).expand()
pop_a

Psi_+*conjugate(Psi_+)/4 + Psi_+*exp(-I*Δϕ)*conjugate(Psi_-)/4 + Psi_-*exp(I*Δϕ)*conjugate(Psi_+)/4 + Psi_-*conjugate(Psi_-)/4

In [18]:
overlap_subs = {
    Ψa * Ψa.conjugate(): 1,
    Ψb * Ψb.conjugate(): 1,
    Ψa * Ψb.conjugate(): symbols("⟨Ψ_{-}|Ψ_{+}⟩"),
    Ψa.conjugate() * Ψb: symbols("⟨Ψ_{-}|Ψ_{+}⟩").conjugate()
}

In [19]:
pop_a.subs(overlap_subs)

⟨Ψ_{-}|Ψ_{+}⟩*exp(-I*Δϕ)/4 + exp(I*Δϕ)*conjugate(⟨Ψ_{-}|Ψ_{+}⟩)/4 + 1/2

For the adiabatic case:

In [20]:
pop_a.subs({Ψa: Ψ0, Ψb:Ψ0}).subs({Ψ0**2:1}).rewrite(exp, cos)

cos(Δϕ)/2 + 1/2

In [21]:
(cos(Δϕ/2)**2).rewrite(cos, exp).expand().rewrite(exp, cos)

cos(Δϕ)/2 + 1/2

## Amplitude and population for "-" surface

In [22]:
b = Ψ_recomb[1,0]

b

-I*Psi_+*exp(-I*Δϕ/2)/2 + I*Psi_-*exp(I*Δϕ/2)/2

Which corresponds to the population

In [23]:
pop_b = (b * b.conjugate()).expand()
pop_b

Psi_+*conjugate(Psi_+)/4 - Psi_+*exp(-I*Δϕ)*conjugate(Psi_-)/4 - Psi_-*exp(I*Δϕ)*conjugate(Psi_+)/4 + Psi_-*conjugate(Psi_-)/4

In [24]:
pop_b.subs(overlap_subs)

-⟨Ψ_{-}|Ψ_{+}⟩*exp(-I*Δϕ)/4 - exp(I*Δϕ)*conjugate(⟨Ψ_{-}|Ψ_{+}⟩)/4 + 1/2

In [25]:
pop_b.subs({Ψa: Ψ0, Ψb:Ψ0}).subs({Ψ0**2:1}).rewrite(exp, cos)

1/2 - cos(Δϕ)/2

In [26]:
(sin(Δϕ/2)**2).rewrite(sin, exp).expand().rewrite(exp, cos)

1/2 - cos(Δϕ)/2